# XGBoost path — one asset end-to-end (NVDA)

Rebuilds the **1-hour XGBoost pipeline** for one asset, exactly as every sealed row in
`xgb/data/oos_metrics.db` was produced — thin orchestration over the committed engine, zero
re-implementation. It is thin orchestration over the
committed engine (`xgb/src/pipeline.py` + `xgb/src/asset_writers.py`) — no logic is re-implemented
here, so what you read is what actually ran.

**The path (layers L4 → L9):**

| Layer | What happens |
|---|---|
| L4 | DuckDB bar snapshot → clean 1h parquet + deterministic 1d / 1w roll-ups (fail-closed source QC) |
| L5 | warmup / Train / OOS time split, with purge (= label horizon) and embargo at every boundary |
| L6 | causal entry candidates + asymmetric ATR Triple-Barrier labels → the "Output B" feature matrix |
| L7 | Optuna tunes XGBoost by **Train out-of-fold trading log-growth** at full-capital sizing; θ fixed at 0.60 |
| L8 | final fit on full Train → self-contained `strategy_NVDA.py` artifact (base64 booster + selfcheck) |
| L9 | the **one-shot OOS read** (2024 → 2026), HODL fallback on zero trades, one metrics row |

Honest framing: over the strong 2024→2026 bull window the strategy does **not** beat buy & hold on
most assets — the demonstrated product is the leak-free *method*. NVDA's sealed one-shot verdict is
**degenerate on purpose** (1 trade, −56.4 %): the raw-price split cliff dominates the path. The final
cell proves this very notebook reproduces that sealed row to numerical tolerance.

> **Why NVDA is a deliberately hard example.** Prices in this project are **raw** (corporate actions deferred), so NVDA's 2024 **10:1 split shows up as a price cliff** inside the OOS window. A split-crossing ticker's absolute numbers are *path arithmetic, not economics* — the project flags this everywhere instead of hiding it, and NVDA's sealed rows are kept as the honest edge case.

## 00 — Setup

Two environment redirects happen **before** the engine import:

- `LIORA_DB` → the **committed** demo bar store (`xgb/data/bars_demo.duckdb`, ~18k 1h NVDA bars,
  2016 → 2026), so the notebook runs from a fresh clone — the full 160 MB store is local-only.
- `OOS_METRICS_DB` → a gitignored scratch db, so running this notebook can **never mutate the
  sealed** `oos_metrics.db`.

Determinism: one seed (`P.seed_everything()`), pinned library set (`requirements-train.txt`).

In [1]:
import os
import shutil
import sys
from pathlib import Path

REPO = Path.cwd().resolve().parent            # this notebook lives in Jupyter-Notebooks/
SRC = REPO / "xgb" / "src"
assert SRC.is_dir(), f"xgb pipeline not found at {SRC} — start jupyter from Jupyter-Notebooks/"

# BOTH redirects must happen BEFORE the pipeline import.
os.environ["LIORA_DB"] = str(REPO / "xgb" / "data" / "bars_demo.duckdb")
os.environ["OOS_METRICS_DB"] = str(REPO / "xgb" / "data" / "verify_scratch.db")
Path(os.environ["OOS_METRICS_DB"]).unlink(missing_ok=True)

os.chdir(SRC)                                 # engine modules import as flat names from src/
sys.path.insert(0, str(SRC))
import pipeline as P
import asset_writers as W

TICKER = "NVDA"
DB_PATH = P.bars_db()

ASSET_DIR = P.REPO_ROOT / "Assets" / TICKER   # regenerable scratch (gitignored)
shutil.rmtree(ASSET_DIR, ignore_errors=True)
ASSET_DIR.mkdir(parents=True, exist_ok=True)
AF = {
    "parquet":    ASSET_DIR / f"{TICKER}_ohlcv_1h.parquet",
    "parquet_1d": ASSET_DIR / f"{TICKER}_ohlcv_1d.parquet",
    "parquet_1w": ASSET_DIR / f"{TICKER}_ohlcv_1w.parquet",
    "optuna":     ASSET_DIR / "OPTUNAs_XGB_HPOs_best_params.json",
    "strategy":   ASSET_DIR / f"strategy_{TICKER}.py",
    "readme":     ASSET_DIR / f"{TICKER}_README.md",
}
P.validate_parameters()
SEED = P.seed_everything()
MANIFEST = P.resolve_feature_manifest(TICKER)
THRESH = P.PIPELINE_PARAMETERS["THRESHOLD_ENTRY"]
CAPITAL_MODE = P.PIPELINE_PARAMETERS["CAPITAL_MODE"]
print("TICKER", TICKER, "| DB", DB_PATH, "| features", MANIFEST["effective_feature_count"])
print("namespaces", MANIFEST["active_namespaces"])

TICKER NVDA | DB ./xgb/data/bars_demo.duckdb | features 33
namespaces ['1h', '1w']


## L4 — DuckDB snapshot → clean 1h parquet, then 1d / 1w roll-up

The raw store is queried once per asset; source QC is **fail-closed** (guard G.1: monotone
timestamps, positive prices, no duplicate bars — a violation raises instead of being cleaned).
Coarser timeframes are *derived deterministically* from the same 1h bars (ET calendar day /
ISO week), so no second data source can leak in.

In [2]:
df = P.layer4_snapshot_to_parquet(DB_PATH, TICKER, AF["parquet"])
print("L4 clean 1h rows:", len(df), "|", df["timestamp"].min(), "->", df["timestamp"].max())

L4 clean 1h rows: 18209 | 2016-01-04 14:00:00+00:00 -> 2026-05-22 19:00:00+00:00


In [3]:
P.layer4_materialize_timeframes(df, {"1d": AF["parquet_1d"], "1w": AF["parquet_1w"]})
import pandas as pd
for tf in ("1d", "1w"):
    d = pd.read_parquet(AF[f"parquet_{tf}"])
    print(f"L4 {tf} rows:", len(d), "| cols:", list(d.columns))

L4 1d rows: 2612 | cols: ['timestamp', 'open', 'high', 'low', 'close', 'volume']
L4 1w rows: 542 | cols: ['timestamp', 'open', 'high', 'low', 'close', 'volume']


## L5 → L6 — time split, feature namespaces, Triple-Barrier Y

- **L5 split:** warmup / Train / OOS boolean masks + integer bounds. Any training event whose
  Triple-Barrier horizon crosses a boundary is **purged** (purge = H) and an **embargo** gap
  follows — the standard defense against label overlap. The OOS window stays frozen until L9.
- **L6 candidates + label:** entry side = `sign(log_return_5)` behind a causal significant-move
  `ENTRY_GATE`; the label is an **asymmetric ATR Triple Barrier** (TP 2×ATR14 / SL 1×ATR14,
  horizon H = 24 bars), `Y = net-positive after costs`.
- **Features are namespaced by timeframe** (1h: 01–99, 1d: 101–199, 1w: 201–299, multi-tf:
  901–999); the 1d/1w context joins by close-time availability (`merge_asof` backward), so an
  in-progress coarse bar can never leak into an intraday decision. The result is the "Output B"
  matrix: one row per candidate event.

In [4]:
REC = P.derive_output_b(df, TICKER, MANIFEST)
print("train candidates:", len(REC["train_events"]), "| Output-B rows:", len(REC["df_b"]))
print("bounds:", REC["bounds"])
print("audit:", REC["audit"])

./xgb/src/pipeline.py:555: RuntimeWarning: Mean of empty slice
  out["momentum_alignment_multi"] = np.nanmean(momentum, axis=0)
./xgb/src/pipeline.py:558: RuntimeWarning: Mean of empty slice
  out["macd_hist_alignment_multi"] = np.nanmean(macd, axis=0)
./xgb/src/pipeline.py:563: RuntimeWarning: Mean of empty slice
  out["price_vs_sma_alignment_multi"] = np.nanmean(sma, axis=0)


train candidates: 9619 | Output-B rows: 9619
bounds: {'train_start_idx': 1393, 'train_end_idx': 14026, 'oos_start_idx': 14027, 'oos_end_idx': 18208}
audit: {'candidates': {'candidates': 9664}, 'eligibility': {'core_nan_excluded': 0, 'core_inf_excluded': 0}, 'scoring': {'train_core_ineligible_skipped': None, 'oos_core_ineligible_skipped': None}}


## L7 — Optuna best hyperparameters (full-capital scoring)

Optuna (TPE, fixed seed, MedianPruner) tunes the XGBoost hyper-parameters by maximizing the
model's own **Train out-of-fold trading log-growth** replayed through the real engine — the model
is selected for *profit*, not for a ranking metric (a proxy metric had been selecting models that
ranked well and traded poorly). Sizing is **all-in compounding** — every trade reinvests the
full running capital (no position sizing), so HPO scores exactly the game that will be played;
the entry threshold θ stays fixed at 0.60.

In [5]:
BEST, AP, FOLDS = P.layer7_optuna(REC["df"], REC["df_b"], REC["train_events"], REC["bounds"], SEED, MANIFEST)
print("best:", BEST)
print("cv AUC-PR:", AP, "| folds:", FOLDS)

./.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


best: {'max_depth': 3, 'eta': 0.262516606324193, 'subsample': 0.892797576724562, 'colsample_bytree': 0.8394633936788146, 'min_child_weight': 2.4041677639819286, 'reg_lambda': 1.201975341512912, 'reg_alpha': 0.2904180608409973, 'gamma': 4.330880728874676, 'n_estimators': 196}
cv AUC-PR: 0.39959238147831166 | folds: 4


In [6]:
CAL = (P.calibrate_kelly(REC["df"], REC["df_b"], REC["train_events"], REC["bounds"], BEST, SEED, MANIFEST)
       if CAPITAL_MODE.startswith("kelly") else {"kelly_fraction": None, "theta_entry": THRESH})
BEST["kelly_fraction"] = CAL["kelly_fraction"]
BEST["theta_entry"] = CAL["theta_entry"]
print("operating point: theta =", BEST["theta_entry"], "| kelly_fraction (lambda):", BEST["kelly_fraction"], "| CAPITAL_MODE:", CAPITAL_MODE)

operating point: theta = 0.6 | kelly_fraction (lambda): None | CAPITAL_MODE: all_in_compounding_per_asset


In [7]:
import json
json.dump({"best_params": BEST, "feature_manifest": MANIFEST,
           "features": {"ids": MANIFEST["effective_feature_ids"],
                        "names": MANIFEST["effective_feature_names"],
                        "active_namespaces": MANIFEST["active_namespaces"],
                        "per_namespace": MANIFEST["per_namespace"]},
           "cv_auc_pr": AP, "cv_folds": FOLDS, "n_trials": P.n_trials()},
          open(AF["optuna"], "w"), indent=2)
print("wrote", AF["optuna"].name)

wrote OPTUNAs_XGB_HPOs_best_params.json


## L8 — the XGBoost strategy artifact

Final fit on the full Train window; the booster is base64-embedded into a **self-contained**
`strategy_NVDA.py` (with a golden-vector selfcheck), and a Train-replay acceptance gate is
recorded. Nothing OOS has been read yet.

In [8]:
BOOSTER = P.layer8_train(REC["df_b"], BEST, SEED, MANIFEST)
KF = BEST.get("kelly_fraction") if CAPITAL_MODE.startswith("kelly") else None
tr_scored = P.score_setups(BOOSTER, REC["df"], REC["train_events"], MANIFEST, REC["feature_context"])
tr_summary, _, _ = P.run_engine(REC["df"], tr_scored, REC["bounds"]["train_start_idx"],
                                REC["bounds"]["oos_start_idx"] - 1, BEST.get("theta_entry", THRESH), kelly_fraction=KF)
ACCEPT = P.accept_strategy(tr_summary)
print("train acceptance:", ACCEPT)

train acceptance: {'accepted': True, 'mode': 'correctness_check', 'rankable': False, 'reason': 'MIN_TRAIN_ACCEPTANCE_TRADES is null -> correctness-check acceptance'}


In [9]:
sp = P.PIPELINE_PARAMETERS["splits"]
META = P.strategy_meta(BOOSTER, REC["df_b"], TICKER, BEST, {},
                       {"start": sp["train_start"], "end": sp["train_end"]}, MANIFEST)
META["ACCEPTANCE"] = ACCEPT
W.write_strategy(AF["strategy"], META)
print("wrote", AF["strategy"].name)

wrote strategy_NVDA.py


## L9 — the one-shot OOS verdict

The 2024 → 2026 window is generated, scored and replayed through the engine **exactly once** —
it never feeds back into any choice above. If the gate produces zero trades, the honest fallback
is buy-and-hold over the same window (flagged as such in the metrics row).

In [10]:
oos_events, _ = P.generate_candidate_events(df, REC["masks"]["oos"], REC["feature_context"])
oos_scored = P.score_setups(BOOSTER, df, oos_events, MANIFEST, REC["feature_context"])
SUMM, LEDGER, _ = P.run_engine(df, oos_scored, REC["bounds"]["oos_start_idx"],
                               REC["bounds"]["oos_end_idx"], BEST.get("theta_entry", THRESH), kelly_fraction=KF)
print("OOS:", {k: SUMM[k] for k in ("start_capital", "end_capital", "profit_factor", "trades")})

if SUMM["trades"] == 0:
    SUMM, LEDGER, _ = P.hodl_fallback(df, REC["bounds"]["oos_start_idx"], REC["bounds"]["oos_end_idx"])
    print("OOS model trades = 0 -> HODL fallback:",
          {k: SUMM[k] for k in ("start_capital", "end_capital", "return_pct", "trades")})

OOS: {'start_capital': 1000.0, 'end_capital': 1000.0, 'profit_factor': None, 'trades': 0}
OOS model trades = 0 -> HODL fallback: {'start_capital': 1000.0, 'end_capital': 436.319799543524, 'return_pct': -56.368020045647604, 'trades': 1}


In [11]:
W.write_readme(AF["readme"], {**SUMM, "ticker": TICKER,
                              "capital_mode": SUMM.get("capital_mode", CAPITAL_MODE),
                              "kelly_fraction": None if SUMM.get("hodl_fallback") else BEST.get("kelly_fraction")},
               LEDGER, MANIFEST)
print("wrote", AF["readme"].name)

OOS_DB = Path(P.oos_db())  # redirected to the scratch db in cell 00 — the sealed store is untouched
W.write_oos_metrics(OOS_DB, {**SUMM, "ticker": TICKER, "cv_auc_pr": AP, "cv_folds": FOLDS,
                             "oos_window": f'{sp["oos_start"]} -> {sp["oos_end"]}'})
print("L9 ->", OOS_DB, ":", TICKER)

wrote NVDA_README.md
L9 -> ./xgb/data/verify_scratch.db : NVDA


In [12]:
six = [AF["parquet"], AF["parquet_1d"], AF["parquet_1w"], AF["optuna"], AF["strategy"], AF["readme"]]
missing = [p.name for p in six if not p.exists()]
assert not missing, f"missing deliverables: {missing}"
print("all 6 engine deliverables present in", ASSET_DIR, ":")
print(sorted(p.name for p in ASSET_DIR.iterdir() if p.is_file()))

all 6 engine deliverables present in ./xgb/Assets/NVDA :
['NVDA_README.md', 'NVDA_ohlcv_1d.parquet', 'NVDA_ohlcv_1h.parquet', 'NVDA_ohlcv_1w.parquet', 'OPTUNAs_XGB_HPOs_best_params.json', 'strategy_NVDA.py']


## Reproducibility check — this run vs the sealed row

The row this notebook just wrote (scratch db) must equal NVDA's **committed sealed row** in
`xgb/data/oos_metrics.db` to `1e-6` relative tolerance — the same check `make verify-xgb` runs.
If this cell passes, every number above is a faithful re-derivation of the sealed result.

In [13]:
import sqlite3

def row(db, t):
    con = sqlite3.connect(f"file:{db}?mode=ro", uri=True)
    con.row_factory = sqlite3.Row
    try:
        r = con.execute("select * from oos_metrics where ticker=?", (t,)).fetchone()
        return dict(r) if r else None
    finally:
        con.close()

sealed = row(P.REPO_ROOT / "data" / "oos_metrics.db", TICKER)
repro = row(Path(os.environ["OOS_METRICS_DB"]), TICKER)
assert sealed and repro, "missing row (sealed or reproduced)"

FIELDS, TOL = ["end_capital", "return_pct", "trades", "profit_factor", "cv_auc_pr"], 1e-6

def close(a, b):
    if a is None or b is None:
        return a == b
    return abs(float(a) - float(b)) <= TOL + TOL * abs(float(b))

diffs = {f: (sealed.get(f), repro.get(f)) for f in FIELDS if not close(sealed.get(f), repro.get(f))}
assert not diffs, f"STALE vs the sealed row: {diffs}"
print("REPRODUCED the sealed NVDA row within 1e-6 tolerance:")
print({f: sealed[f] for f in FIELDS})

REPRODUCED the sealed NVDA row within 1e-6 tolerance:
{'end_capital': 436.319799543524, 'return_pct': -56.368020045647604, 'trades': 1, 'profit_factor': 0.0, 'cv_auc_pr': 0.39959238147831166}
